# ニューラルネットを、NumPy だけで書く

ライブラリに任せず、**自分で式を書きます**。

1 個の計算点から始めて、XOR で行き詰まり、層を重ね、誤差逆伝播で学習させるまで。
Arc 2 第 1 話「冬を越えた者たち」で起きたことを、そのまま手でたどります。

途中で、書いた勾配が正しいかを数値微分で検算します。

- **戻る**: [ニューラルネット — 計算と学習の仕組み](https://manga-epoch.github.io/viewer/pub/epoch/arc2/figures_nn.html) — 同じ内容をスライダで動かせます
- **必要なもの**: NumPy と matplotlib のみ（Colab の標準環境でそのまま動きます）
- 上から順に実行してください（Colab では `Shift + Enter`）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## 1. 1 個の計算点

入力に重みを掛けて足し、しきい値を超えたら 1 を出す。これだけです。

$$y=\begin{cases}1 & (w\cdot x + b > 0)\\ 0 & (\text{それ以外})\end{cases}$$

In [ ]:
def perceptron(x, w, b):
    return (x @ w + b > 0).astype(int)

X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
y_and = np.array([0, 0, 0, 1])

w = np.array([1.0, 1.0]); b = -1.5      # 手で置いた重み
print("AND を手で作る")
for x, t in zip(X_and, y_and):
    print(f"  {x} → {perceptron(x, w, b)}  (正解 {t})")

### 重みを、学習で決める

手で置くのではなく、間違えたぶんだけ動かします。

$$w \leftarrow w + \eta\,(t-y)\,x, \qquad b \leftarrow b + \eta\,(t-y)$$

$t$ が正解、$y$ が出力。**合っていれば $t-y=0$ で何も動かない**のがこの式の要点です。

In [ ]:
def train_perceptron(X, y, epochs=20, lr=0.1):
    w = np.zeros(X.shape[1]); b = 0.0
    hist = []
    for _ in range(epochs):
        wrong = 0
        for xi, ti in zip(X, y):
            yi = int(xi @ w + b > 0)
            w += lr * (ti - yi) * xi
            b += lr * (ti - yi)
            wrong += int(ti != yi)
        hist.append(wrong)
    return w, b, hist

w, b, hist = train_perceptron(X_and, y_and)
print(f"学習後: w={w.round(2)}, b={b:.2f}")
print("各周の間違い数:", hist)
print("出力:", perceptron(X_and, w, b), " 正解:", y_and)

## 2. XOR で行き詰まる

XOR（違うときだけ 1）を同じやり方で学習させます。

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
y_xor = np.array([0, 1, 1, 0])

w, b, hist = train_perceptron(X_xor, y_xor, epochs=200)
print("各周の間違い数（最後の 20 周）:", hist[-20:])
print("出力:", perceptron(X_xor, w, b), " 正解:", y_xor)
print("\n何周まわしても間違いが 0 になりません。")

In [ ]:
# なぜ無理なのか。1 個の計算点が引けるのは「1 本の直線」だけで、
# XOR の ○ と × は 1 本の直線では分けられません。
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (y, name) in zip(axes, [(y_and, "AND (separable)"), (y_xor, "XOR (not separable)")]):
    for cls, mk, col in [(0, "o", "#6E7BA8"), (1, "x", "#B4493F")]:
        pts = X_xor[y == cls]
        ax.scatter(pts[:, 0], pts[:, 1], marker=mk, s=140, c=col, linewidths=2)
    ax.set_title(name); ax.set_xlim(-.4, 1.4); ax.set_ylim(-.4, 1.4); ax.grid(alpha=.3)
axes[0].plot([-.4, 1.4], [1.6, -.2], "k--", lw=1)     # AND を分ける直線の一例
plt.tight_layout(); plt.show()

1969 年、ミンスキーとパパートがこれを指摘し、研究への出資は細りました
（Arc 2 第 1 話の背景です）。**答えは「層を重ねること」**でしたが、
重ねた層をどう学習させるかが分からない期間が続きました。

## 3. 層を重ねる

段差のある階段（しきい値）では微分ができず、学習の手がかりが作れません。
なめらかな関数に取り替えます。

$$\sigma(z)=\frac{1}{1+e^{-z}},\qquad \sigma'(z)=\sigma(z)\,(1-\sigma(z))$$

2 層にすると、順伝播はこうなります。

$$h=\sigma(xW_1+b_1),\qquad \hat{y}=\sigma(hW_2+b_2)$$

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def init(n_in, n_hid, n_out, scale=1.0):
    return {
        "W1": np.random.randn(n_in, n_hid) * scale, "b1": np.zeros(n_hid),
        "W2": np.random.randn(n_hid, n_out) * scale, "b2": np.zeros(n_out),
    }

def forward(p, X):
    z1 = X @ p["W1"] + p["b1"]; h = sigmoid(z1)
    z2 = h @ p["W2"] + p["b2"]; o = sigmoid(z2)
    return h, o

def loss(p, X, y):
    _, o = forward(p, X)
    return float(np.mean((o.ravel() - y) ** 2))

p = init(2, 4, 1)
print("学習前の損失:", round(loss(p, X_xor, y_xor), 4))

## 4. 誤差逆伝播

出力の誤差を、**連鎖律で 1 層ずつ手前へ渡します**。

$$\delta_2=(\hat{y}-y)\odot\sigma'(z_2), \qquad
\frac{\partial L}{\partial W_2}=h^{\top}\delta_2$$

$$\delta_1=(\delta_2 W_2^{\top})\odot\sigma'(z_1), \qquad
\frac{\partial L}{\partial W_1}=x^{\top}\delta_1$$

$\odot$ は要素ごとの積です。$\sigma'$ を掛ける操作が各層に入る点が、
あとで効いてきます（節 6）。

In [ ]:
def backward(p, X, y):
    n = len(X)
    z1 = X @ p["W1"] + p["b1"]; h = sigmoid(z1)
    z2 = h @ p["W2"] + p["b2"]; o = sigmoid(z2)
    d2 = (o - y.reshape(-1, 1)) * o * (1 - o) * (2.0 / n)
    d1 = (d2 @ p["W2"].T) * h * (1 - h)
    return {"W2": h.T @ d2, "b2": d2.sum(0), "W1": X.T @ d1, "b1": d1.sum(0)}

### 検算 — この勾配は本当に正しいか

逆伝播は符号や転置を 1 つ間違えても「それらしく」学習が進んでしまい、
気づけません。**数値微分**と突き合わせます。

$$\frac{\partial L}{\partial\theta}\approx\frac{L(\theta+\varepsilon)-L(\theta-\varepsilon)}{2\varepsilon}$$

これは定義そのものなので、必ず正しい値が出ます（ただし遅い）。

In [ ]:
def numerical_grad(p, X, y, key, eps=1e-5):
    g = np.zeros_like(p[key])
    it = np.nditer(p[key], flags=["multi_index"])
    while not it.finished:
        i = it.multi_index
        old = p[key][i]
        p[key][i] = old + eps; a = loss(p, X, y)
        p[key][i] = old - eps; b = loss(p, X, y)
        p[key][i] = old
        g[i] = (a - b) / (2 * eps)
        it.iternext()
    return g

p = init(2, 4, 1)
mine = backward(p, X_xor, y_xor)
print("手で書いた勾配 vs 数値微分")
ok = True
for key in ["W1", "b1", "W2", "b2"]:
    num = numerical_grad(p, X_xor, y_xor, key)
    rel = np.abs(mine[key] - num).max() / (np.abs(num).max() + 1e-12)
    ok &= rel < 1e-5
    print(f"  {key}: 相対誤差 {rel:.2e}")
print("\n→ 一致。" if ok else "\n→ 不一致。式を見直してください。")

## 5. 学習させる

勾配の向きと逆に、少しずつ動かします。$\eta$ が学習率です。

$$\theta \leftarrow \theta - \eta\,\frac{\partial L}{\partial \theta}$$

In [ ]:
def train(p, X, y, epochs=8000, lr=1.0):
    hist = []
    for _ in range(epochs):
        g = backward(p, X, y)
        for k in p:
            p[k] -= lr * g[k]
        hist.append(loss(p, X, y))
    return hist

p = init(2, 4, 1)
hist = train(p, X_xor, y_xor)
_, o = forward(p, X_xor)
print("出力:", o.ravel().round(3), " 正解:", y_xor)
print("損失:", round(hist[-1], 5))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist); axes[0].set_yscale("log")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss (log)"); axes[0].grid(alpha=.3)

gx, gy = np.meshgrid(np.linspace(-.3, 1.3, 200), np.linspace(-.3, 1.3, 200))
grid = np.c_[gx.ravel(), gy.ravel()]
axes[1].contourf(gx, gy, forward(p, grid)[1].reshape(gx.shape), levels=20, cmap="RdBu_r")
for cls, mk, col in [(0, "o", "#1a1a1a"), (1, "x", "#1a1a1a")]:
    pts = X_xor[y_xor == cls]
    axes[1].scatter(pts[:, 0], pts[:, 1], marker=mk, s=160, c=col, linewidths=2.5)
axes[1].set_title("decision boundary (XOR)")
plt.tight_layout(); plt.show()

1 本の直線では引けなかった境界が、**曲がった形**で引けています。
隠れ層の 4 個が、それぞれ 1 本ずつ直線を引き、出力層がそれを組み合わせた結果です。

In [ ]:
# 隠れ層の 1 個ずつが何を見ているか。4 枚とも「1 本の直線」です
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for j, ax in enumerate(axes):
    hj = sigmoid(grid @ p["W1"] + p["b1"])[:, j].reshape(gx.shape)
    ax.contourf(gx, gy, hj, levels=20, cmap="Greys")
    ax.set_title(f"hidden unit {j+1}", fontsize=10)
plt.tight_layout(); plt.show()

## 6. なぜ「深く」できなかったのか — 勾配消失

逆伝播は層をさかのぼるたびに $\sigma'$ を掛けます。
$\sigma'$ の最大値は $0.25$ なので、$L$ 層さかのぼると最大でも $0.25^{L}$ 倍。
**手前の層に届くころには、ほとんど 0 になります。**

実際に測ります。

In [ ]:
def deep_grad_scale(n_layers, act="sigmoid", n=64, dim=32, seed=0):
    rng = np.random.default_rng(seed)
    Ws = [rng.standard_normal((dim, dim)) * (1.0 / np.sqrt(dim)) for _ in range(n_layers)]
    x = rng.standard_normal((n, dim))
    hs, zs = [x], []
    for W in Ws:
        z = hs[-1] @ W; zs.append(z)
        hs.append(sigmoid(z) if act == "sigmoid" else np.maximum(0, z))
    d = np.ones_like(hs[-1]) / n
    scales = []
    for i in reversed(range(n_layers)):
        d = d * (hs[i+1] * (1 - hs[i+1]) if act == "sigmoid" else (zs[i] > 0).astype(float))
        scales.append(np.abs(hs[i].T @ d).mean())
        d = d @ Ws[i].T
    return scales[::-1]        # 入力側 → 出力側

L = 20
for act in ["sigmoid", "relu"]:
    s = deep_grad_scale(L, act)
    print(f"{act:8} 第1層 {s[0]:.3e} / 第{L}層 {s[-1]:.3e} → 比 {s[0]/s[-1]:.2e}")

plt.figure(figsize=(7, 4))
for act, c in [("sigmoid", "#B4493F"), ("relu", "#6E7BA8")]:
    plt.plot(range(1, L+1), deep_grad_scale(L, act), "o-", color=c, label=act)
plt.yscale("log"); plt.xlabel("layer (1 = closest to input)")
plt.ylabel("|gradient| (log)"); plt.legend(); plt.grid(alpha=.3)
plt.title("how far the gradient reaches"); plt.show()

シグモイドでは、入力に近い層の勾配が桁で小さくなります。**学習していないのと同じ**です。

ReLU（$\max(0,z)$）は正の側で微分が 1 なので、掛け算で減りません。
2010 年前後にここが変わったことが、深い層を現実的にしました
（Arc 2 第 4 話・図解「[深さの積み方](https://manga-epoch.github.io/viewer/pub/epoch/arc2/figures_depth.html)」）。

## 次に

- **図解に戻る**: [ニューラルネット — 計算と学習の仕組み](https://manga-epoch.github.io/viewer/pub/epoch/arc2/figures_nn.html)
- **画像でやる**: [cnn_on_your_image.ipynb](https://colab.research.google.com/github/manga-epoch/viewer/blob/main/notebooks/cnn_on_your_image.ipynb)
- **言語でやる**: [attention.ipynb](https://colab.research.google.com/github/manga-epoch/viewer/blob/main/notebooks/attention.ipynb)

---

## 出典とライセンス

このノートブックは、マンガ **EPOCH — 時代の前夜** の④「書く」レイヤーです。
[EPOCH について](https://manga-epoch.github.io/viewer)

本ノートブックのコードは自由に改変して使えます。